In [1]:
!pip install transformers

In [2]:
!pip install transformers==4.44.2 --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 23.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.19.0
    Uninstalling huggingface_hub-1.19.0:
      Successfully uninstalled huggingface_hub-1.19.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transformers-5.12.0


## Cell 2: Summarizer Agent

In [3]:
from transformers import pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

text = """
This agreement is made between Party A and Party B for providing services for one year.
Payment will be monthly. Either party can terminate with 30 days notice.
"""

summary = summarizer(
    text,
    max_length=60,
    min_length=25,
    do_sample=False
)

print(summary[0]['summary_text'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Your max_length is set to 60, but your input_length is only 38. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=19)


This agreement is made between Party A and Party B for providing services for one year.Payment will be monthly. Either party can terminate with 30 days notice.


## Adding Chunk

In [4]:
def chunk_text(text, max_words=300):
    words = text.split()
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

## Chunk Summarizer

In [5]:
chunks = chunk_text(text)
summaries = [summarizer(c, max_length=60, min_length=25)[0]['summary_text'] for c in chunks]

final_summary = " ".join(summaries)
print(final_summary)

Your max_length is set to 60, but your input_length is only 33. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=16)


This agreement is made between Party A and Party B for providing services for one year. Payment will be monthly. Either party can terminate with 30 days notice.


## Classifier Code

In [6]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

text = """
This agreement is made between Party A and Party B for providing services.
The payment will be made monthly and the contract duration is one year.
"""

labels = [
    "legal contract agreement",
    "first information report (police complaint)",
    "property lease or rental agreement",
    "affidavit sworn statement",
    "legal notice or warning document"
]

result = classifier(
    text,
    labels,
    hypothesis_template="This document is a {}."
)

print(result)
print("Predicted Document Type:", result['labels'][0])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': '\nThis agreement is made between Party A and Party B for providing services.\nThe payment will be made monthly and the contract duration is one year.\n', 'labels': ['legal contract agreement', 'legal notice or warning document', 'property lease or rental agreement', 'affidavit sworn statement', 'first information report (police complaint)'], 'scores': [0.7360785603523254, 0.10097994655370712, 0.06690319627523422, 0.059066373854875565, 0.03697190806269646]}
Predicted Document Type: legal contract agreement


## Risk Analyzer

In [7]:
import re

def risk_analyzer(text):
    text_lower = text.lower()
    risks = []

    high_risk_patterns = [
        r"all losses",
        r"without limitation",
        r"unlimited liability",
        r"indemnif\w+",
        r"held liable for any damages"
    ]

    medium_risk_patterns = [
        r"terminate anytime",
        r"no notice",
        r"penalty",
        r"breach",
        r"fine"
    ]

    for pattern in high_risk_patterns:
        if re.search(pattern, text_lower):
            risks.append(("HIGH", pattern))

    for pattern in medium_risk_patterns:
        if re.search(pattern, text_lower):
            risks.append(("MEDIUM", pattern))

    return risks


text = """
The supplier shall be liable for all losses without limitation.
Either party can terminate anytime without notice.
"""

print(risk_analyzer(text))

[('HIGH', 'all losses'), ('HIGH', 'without limitation'), ('MEDIUM', 'terminate anytime')]


## Drafting Agent

In [8]:
from transformers import pipeline

draft_model = pipeline("text2text-generation", model="google/flan-t5-base")


def drafting_agent(clause, risk_level="MEDIUM"):

    prompt = f"""
You are a legal drafting assistant.

Your task is to rewrite the following legal clause to make it safer, clearer, and more balanced.

Rules:
- Preserve original meaning
- Remove unfair or risky terms
- Make it legally fair for both parties
- Keep professional legal tone

Risk Level: {risk_level}

Clause:
{clause}

Rewrite:
"""

    result = draft_model(
        prompt,
        max_length=200,
        do_sample=False
    )

    return result[0]['generated_text']

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

## Testing with risk analyzer

In [9]:
clause = "The supplier shall be liable for all losses without limitation."

print(drafting_agent(clause, "HIGH"))

The supplier shall be liable for all losses without limitation.


## Version 2 (Integrated with risk system)

In [10]:
def drafting_agent(clause, risk_info):

    prompt = f"""
You are a legal document rewriting assistant.

Rewrite the clause to reduce legal risk while preserving meaning.

Risk information:
{risk_info}

Guidelines:
- HIGH risk → strongly modify clause
- MEDIUM risk → soften language
- LOW risk → minor improvements only

Clause:
{clause}

Provide:
1. Revised Clause
2. Short Explanation
"""

    result = draft_model(
        prompt,
        max_length=220,
        do_sample=False
    )

    return result[0]['generated_text']

## Example integration

In [11]:
text = "The supplier shall be liable for all losses without limitation."

risk = risk_analyzer(text)

output = drafting_agent(text, risk)

print(output)

1. The supplier shall be liable for all losses without limitation.


In [2]:
git init
git remote add origin https://github.com/HashmatHashmi/Legal_Document_Analysis_Agent

SyntaxError: invalid syntax (3853649345.py, line 1)